In [26]:
import pandas as pd
import re
from collections import Counter

# Define a list of SQL operations we want to track
operations = ['SELECT', 'FROM', 'JOIN', 'WHERE', 'GROUP BY', 'ORDER BY', 'HAVING', 'LIMIT', 'INSERT', 'UPDATE', 'DELETE']

def query_len(query):
    return len(query)

def count_sql_operations(query):
    # Convert the query to uppercase to handle case insensitivity
    query = query.upper()

    # Initialize a counter to track operation counts
    operation_counts = Counter()

    # Loop through each operation type and count its occurrences
    for operation in operations:
        # Find all occurrences of each operation
        matches = re.findall(r'\b' + re.escape(operation) + r'\b', query)
        operation_counts[operation] = len(matches)

    return operation_counts

In [27]:
########################################################################
######### CHANGE INPUT FOLDER HERE #####################################
########################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7.xlsx"
# Read the excel file
df = pd.read_excel(file_path, engine='openpyxl')
df.head()

,db_id,spider_query,question,text2sql_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;"
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;"
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""..."
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O..."
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...


In [28]:
for index, row in df.iterrows():

    print("Number of row: ",index)
    
    target_query = row['spider_query']
    predicted_query = row['text2sql_query']

    # get query length
    target_len = query_len(target_query)

    # Store the len for query 1 in the DataFrame
    df.at[index, 'spider_query_length'] = str(target_len)
    print("Length of target_query:", target_len)

    # get query length
    predicted_len = query_len(predicted_query)

    # Store the len for query 2 in the DataFrame
    df.at[index, 'text2sql_query_length'] = str(predicted_len)
    print("Length of predicted_query:", predicted_len)
    
    # Compare the length
    if predicted_len == target_len:
        df.at[index, 'hallucination'] = False
    else:

        predicted_operations_counts = count_sql_operations(predicted_query)
        target_operations_counts = count_sql_operations(target_query)
        # print(predicted_operations_counts,predicted_query,target_operations_counts,target_query)
        
    # Matching counts
    matching_counts = sum(1 for key in predicted_operations_counts if predicted_operations_counts[key] == target_operations_counts[key])
    
    # Calculate the total number of clauses (operations)
    total_clauses = len(predicted_operations_counts)
    
    # Calculate the similarity score as the percentage of matching counts
    similarity_score = matching_counts / total_clauses
    
    # Print the results
    print(f"Matching Clause Counts: {matching_counts}")
    print(f"Total Clauses: {total_clauses}")
    print(f"Similarity Score: {similarity_score:.2f}")
    
    # Optionally, you can also print the differences:
    difference = predicted_operations_counts - target_operations_counts
    for ops in operations:
        try:
            ops_value = dict(difference)[ops]
        except:
            ops_value = 0
            
        df.at[index, ops+'__difference'] = ops_value
        
    if (len(dict(difference).values()) > 0) and (max(dict(difference).values()) > 5) :
        df.at[index, 'hallucination'] = True
    else:
        df.at[index, 'hallucination'] = False
    print("#"*50)

Number of row:  0
Length of target_query: 25
Length of predicted_query: 47
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  1
Length of target_query: 25
Length of predicted_query: 49
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  2
Length of target_query: 55
Length of predicted_query: 91
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  3
Length of target_query: 55
Length of predicted_query: 73
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  4
Length of target_query: 59
Length of predicted_query: 76
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  5
Length of ta

Matching Clause Counts: 9
Total Clauses: 11
Similarity Score: 0.82
##################################################
Number of row:  224
Length of target_query: 65
Length of predicted_query: 104
Matching Clause Counts: 10
Total Clauses: 11
Similarity Score: 0.91
##################################################
Number of row:  225
Length of target_query: 65
Length of predicted_query: 104
Matching Clause Counts: 10
Total Clauses: 11
Similarity Score: 0.91
##################################################
Number of row:  226
Length of target_query: 107
Length of predicted_query: 92
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  227
Length of target_query: 107
Length of predicted_query: 91
Matching Clause Counts: 9
Total Clauses: 11
Similarity Score: 0.82
##################################################
Number of row:  228
Length of target_query: 136
Length of predicted_query: 133
Matching Clause 

Matching Clause Counts: 9
Total Clauses: 11
Similarity Score: 0.82
##################################################
Number of row:  449
Length of target_query: 368
Length of predicted_query: 320
Matching Clause Counts: 9
Total Clauses: 11
Similarity Score: 0.82
##################################################
Number of row:  450
Length of target_query: 110
Length of predicted_query: 117
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  451
Length of target_query: 110
Length of predicted_query: 117
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  452
Length of target_query: 88
Length of predicted_query: 111
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  453
Length of target_query: 88
Length of predicted_query: 111
Matching Claus

##################################################
Number of row:  697
Length of target_query: 45
Length of predicted_query: 50
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  698
Length of target_query: 45
Length of predicted_query: 52
Matching Clause Counts: 11
Total Clauses: 11
Similarity Score: 1.00
##################################################
Number of row:  699
Length of target_query: 62
Length of predicted_query: 97
Matching Clause Counts: 10
Total Clauses: 11
Similarity Score: 0.91
##################################################
Number of row:  700
Length of target_query: 62
Length of predicted_query: 117
Matching Clause Counts: 9
Total Clauses: 11
Similarity Score: 0.82
##################################################
Number of row:  701
Length of target_query: 68
Length of predicted_query: 87
Matching Clause Counts: 8
Total Clauses: 11
Similarity Score: 0.73
#####################

In [29]:
df.head()

,db_id,spider_query,question,text2sql_query,spider_query_length,text2sql_query_length,SELECT__difference,FROM__difference,JOIN__difference,WHERE__difference,GROUP BY__difference,ORDER BY__difference,HAVING__difference,LIMIT__difference,INSERT__difference,UPDATE__difference,DELETE__difference,hallucination
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(DISTINCT f.Farm_ID) FROM ""farm"" f;",25,47,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
1,farm,SELECT count(*) FROM farm,Count the number of farms.,"SELECT COUNT(DISTINCT f.""Farm_ID"") FROM ""farm"" f;",25,49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Year"", f.""Total_Horses""...",55,91,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.Farm_ID, f.Total_Horses FROM farm f O...",55,73,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...,59,76,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False


In [30]:
#############################################################################
######### CHANGE OUTPUT FOLDER FILE HERE ####################################
#############################################################################
output_file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-Hallucination-Check.xlsx"
df.to_excel(output_file_path, index=False)
print(f"DataFrame saved successfully to {output_file_path}")

DataFrame saved successfully to C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-7-Hallucination-Check.xlsx
